# Ejercicio 2: Análisis y Control frente a Perturbaciones

**Curso:** IOR442 Sistemas de Control — Dr. Mariano Scaramal  
**TP1:** Diseño de Controladores con Root Locus y Respuesta en Frecuencia

---

## Descripción del problema

En el sistema de orientación del rotor, el coeficiente de fricción viscosa $B$ puede degradarse por fatiga térmica, pasando de $B = 0{,}5$ (nominal) a $B = -0{,}1$ (falla). Con $B < 0$, el sistema se vuelve inestable.

### Modelo simplificado

Se desprecia la dinámica eléctrica (constante de tiempo mucho menor que la mecánica):

$$G(s) = \frac{K}{s(Js + B)} = \frac{1}{s(s + B)}, \quad K = 1,\; J = 1$$

### Diagrama de bloques

```
R(s) ──►(+)──► C(s) ──► 1/[s(s+B)] ──► Θ(s)
         -▲                                │
          └────────────────────────────────┘
               realimentación unitaria
```

| Parámetro | Valor nominal | Valor con falla | Unidad |
|-----------|:---:|:---:|:---:|
| $K$ | 1 | 1 | — |
| $J$ | 1 | 1 | — |
| $B$ | 0,5 | −0,1 | — |

In [ ]:
# Importaciones y configuración
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import control as ctrl

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from planta2 import (crear_planta2, crear_planta2_ss, info_planta2,
                     K, J, B_NOM, B_FALLA)
from robustez import (graficar_nyquist, analizar_estabilidad_nyquist,
                      simular_falla, graficar_respuesta_falla,
                      graficar_rlocus_comparativo,
                      disenar_lead_por_margen_fase,
                      comparar_bode_multiples,
                      graficar_nyquist_comparativo)

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 1.5,
})

# Crear plantas
G_nom = crear_planta2(B_NOM)
G_falla = crear_planta2(B_FALLA)

print("Plantas creadas:")
info_planta2(B_NOM)
info_planta2(B_FALLA)

---
# Parte A — Análisis de Sensibilidad y Robustez

---

## A.a) Diagrama de Nyquist — Sistema nominal ($B = 0{,}5$)

Para el sistema nominal $G(s) = \frac{1}{s(s+0{,}5)}$ en lazo cerrado con realimentación unitaria, aplicamos el **criterio de Nyquist**:

$$Z = N + P$$

- $P$: polos de $G(s)$ en el semiplano derecho (SPD)
- $N$: encirculamientos horarios de $(-1, 0)$
- $Z$: polos de lazo cerrado en SPD → para estabilidad, $Z = 0$

In [ ]:
# A.a) Diagrama de Nyquist del sistema nominal (B = 0.5)

fig, count = graficar_nyquist(G_nom, titulo="Nyquist — G(s) nominal (B = 0.5)")
plt.show()

# Análisis de estabilidad
result = analizar_estabilidad_nyquist(G_nom, B_NOM)

### Observaciones A.a)

- El sistema nominal ($B = 0{,}5$) tiene polos en $s = 0$ y $s = -0{,}5$. Ninguno está en el SPD ($P = 0$).
- La curva de Nyquist **no rodea** el punto $(-1, 0)$ → $N = 0$.
- Por lo tanto $Z = N + P = 0$ → el sistema es **estable en lazo cerrado**.

---

## A.b) Simulación de falla ($B$: 0,5 → −0,1 en $t = 15$ s)

Como el parámetro $B$ cambia en el tiempo, el sistema es **lineal variante en el tiempo (LTV)**. No podemos usar funciones de transferencia; usamos espacio de estados con integración numérica (`scipy.integrate.solve_ivp`).

$$\dot{\theta} = \omega, \quad \dot{\omega} = -\frac{B(t)}{J}\omega + \frac{K}{J}(r - \theta)$$

donde $B(t) = \begin{cases} 0{,}5 & t < 15 \\ -0{,}1 & t \geq 15 \end{cases}$

In [ ]:
# A.b) Simulación con falla en t=15s (sin compensar, C=1)

t, theta, omega = simular_falla(C_tf=None, t_falla=15, t_final=50,
                                 B_pre=B_NOM, B_post=B_FALLA)

fig = graficar_respuesta_falla(t, theta,
    titulo="A.b) Respuesta al escalón con falla en t=15s (C=1)")
plt.show()

# Determinar si se vuelve inestable
theta_post = theta[t > 20]
if np.max(np.abs(theta_post)) > 10:
    print("El sistema SE VUELVE INESTABLE después de la falla.")
    print(f"  θ diverge: θ(t={t[-1]:.0f}) = {theta[-1]:.2f}")
else:
    print("El sistema permanece acotado después de la falla.")
    print(f"  θ(t={t[-1]:.0f}) = {theta[-1]:.4f}")

### Observaciones A.b)

- Antes de $t = 15$ s, el sistema sigue el escalón unitario correctamente (estable con $B = 0{,}5$).
- Después de la falla ($B = -0{,}1$), el sistema **se vuelve inestable**: la salida diverge.
- Esto ocurre porque con $B < 0$, la planta tiene un polo en $s = +0{,}1$ (SPD). En lazo cerrado sin compensador, el sistema no tiene suficiente margen para mantener la estabilidad.

---

## A.c) Root Locus comparativo: nominal vs falla

In [ ]:
# A.c) Root Locus comparativo para B=0.5 y B=-0.1

fig = graficar_rlocus_comparativo(
    [B_NOM, B_FALLA],
    labels=[f'B = {B_NOM} (nominal)', f'B = {B_FALLA} (falla)'],
    colores=['b', 'r']
)
plt.title('A.c) Root Locus: nominal (azul) vs falla (rojo)')
plt.show()

# Polos de lazo cerrado (C=1) para cada caso
for B_val, label in [(B_NOM, 'Nominal'), (B_FALLA, 'Falla')]:
    G = crear_planta2(B_val)
    T = ctrl.feedback(G, 1)
    polos_lc = ctrl.poles(T)
    estable = all(np.real(polos_lc) < 0)
    print(f"  {label} (B={B_val}): Polos LC = {polos_lc}")
    print(f"    Estable: {'SÍ' if estable else 'NO'}")

### Observaciones A.c)

- **Nominal ($B = 0{,}5$)**: polos de LA en $s = 0$ y $s = -0{,}5$. El root locus se mueve hacia la izquierda con ganancia → polos de LC estables.
- **Falla ($B = -0{,}1$)**: polos de LA en $s = 0$ y $s = +0{,}1$. El root locus parte del SPD. Para ganancia $K=1$, al menos un polo de LC tiene parte real positiva → **inestable**.
- La ubicación de los polos muestra que el sistema es **muy sensible** a variaciones de $B$: un pequeño cambio de signo destruye la estabilidad.

---

## A.d) Propuesta de tipo de compensador

Se propone un **compensador lead (adelanto de fase)** por las siguientes razones:

1. **Aporta fase positiva** en frecuencias medias → incrementa el margen de fase.
2. Un mayor margen de fase significa que el sistema tolera más degradación de $B$ antes de volverse inestable.
3. Un compensador **lag** reduciría el ancho de banda y empeoraría el transitorio.
4. Un **lead-lag** sería más complejo y no es necesario si no hay requisitos de precisión adicionales.

El efecto del lead sobre la fase es "alejar" la curva de Nyquist del punto crítico $(-1,0)$, otorgando un mayor margen de seguridad.

> **Nota del enunciado**: normalizar la ganancia DC del compensador a 1 para no alterar el error de posición. Esto es posible porque un lead $C(s) = K_c \cdot \frac{s+z}{s+p}$ con $K_c = p/z$ tiene $C(0) = 1$.

---
---

# Parte B — Diseño de Compensador para Estabilidad Robusta

Se diseñan tres compensadores lead para el sistema nominal ($B = 0{,}5$) con distintos márgenes de fase objetivo: **35°, 50° y 65°**.

---

## B.a) Diseño de compensadores lead normalizados

In [ ]:
# B.a) Diseño de tres compensadores lead para PM = 35°, 50°, 65°

PM_objetivos = [35, 50, 65]
compensadores = []
infos = []

for PM in PM_objetivos:
    C, info = disenar_lead_por_margen_fase(G_nom, PM, B_val=B_NOM)
    compensadores.append(C)
    infos.append(info)
    print()

# Tabla resumen
print(f"{'='*70}")
print(f"  {'PM obj':>8} {'PM obt':>8} {'α':>8} {'z':>8} {'p':>8} {'Kc':>8} {'C(0)':>8}")
print(f"{'='*70}")
for info in infos:
    print(f"  {info['PM_deseado']:>6.0f}°  {info['PM_obtenido']:>6.1f}°  "
          f"{info['alpha']:>8.4f} {info['z']:>8.4f} {info['p']:>8.4f} "
          f"{info['Kc']:>8.4f} {info['ganancia_DC']:>8.4f}")
print(f"{'='*70}")
print("\nNota: C(0) ≈ 1 confirma la normalización de ganancia en DC.")
print("Esto es posible porque el lead tiene |C(0)| = z/p < 1,")
print("y Kc = p/z compensa exactamente para que C(0) = 1.")

---

## B.b) Verificación por Bode — Los tres compensadores

In [ ]:
# B.b) Bode comparativo de los 3 compensadores

labels = [f'PM = {info["PM_deseado"]}° (obtenido: {info["PM_obtenido"]:.1f}°)'
          for info in infos]

fig = comparar_bode_multiples(G_nom, compensadores, labels,
    colores=['b', 'r', 'g'],
    titulo="B.b) Bode del sistema nominal compensado — PM = 35°, 50°, 65°")
plt.show()

# Verificar márgenes
print("Márgenes de fase verificados:")
for C, info in zip(compensadores, infos):
    CG = C * G_nom
    gm, pm, wcg, wcp = ctrl.margin(CG)
    gm_dB = 20 * np.log10(gm) if np.isfinite(gm) else float('inf')
    print(f"  PM objetivo = {info['PM_deseado']}° → PM = {pm:.1f}°, GM = {gm_dB:.1f} dB, ωcp = {wcp:.2f} rad/s")

### Observaciones B.b)

- Los tres compensadores logran márgenes de fase cercanos a los objetivos.
- A mayor PM, el **ancho de banda disminuye** ($\omega_{cp}$ menor) → respuesta más lenta pero más robusta.
- La normalización $C(0) = 1$ se verifica en el Bode: la ganancia en baja frecuencia es la misma para todos los casos.

---

## B.c) Respuesta al escalón con falla en $t = 15$ s

In [ ]:
# B.c) Respuesta al escalón con falla para los 3 compensadores

fig, ax = plt.subplots(figsize=(10, 7))

colores = ['b', 'r', 'g']
for C, info, color in zip(compensadores, infos, colores):
    t, theta, omega = simular_falla(C_tf=C, t_falla=15, t_final=50,
                                     B_pre=B_NOM, B_post=B_FALLA)
    ax.plot(t, theta, color=color, linewidth=1.5,
            label=f'PM = {info["PM_deseado"]}°')

# También sin compensar para referencia
t0, theta0, _ = simular_falla(C_tf=None, t_falla=15, t_final=50)
ax.plot(t0, theta0, 'k--', linewidth=1, alpha=0.5, label='Sin compensar')

ax.axhline(1.0, color='k', linestyle=':', linewidth=0.8, alpha=0.3)
ax.axvline(15, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Falla (t=15s)')
ax.set_xlabel('Tiempo [s]')
ax.set_ylabel('$\\theta$ [rad]')
ax.set_title('B.c) Respuesta al escalón con falla (B: 0.5 → −0.1 en t=15s)')
ax.legend(loc='best')
ax.set_xlim([0, 50])
ax.set_ylim([-2, 5])  # limitar para ver bien (sin compensar diverge)
plt.tight_layout()
plt.show()

# Analizar tiempos de establecimiento post-falla
print("Análisis post-falla:")
for C, info, color in zip(compensadores, infos, colores):
    t, theta, omega = simular_falla(C_tf=C, t_falla=15, t_final=100)
    theta_post = theta[t > 20]
    estable = np.max(np.abs(theta_post - 1.0)) < 1.0
    ts_approx = "—"
    if estable:
        banda = np.where(np.abs(theta[t > 15] - 1.0) > 0.02)[0]
        t_post = t[t > 15]
        if len(banda) > 0:
            ts_approx = f"{t_post[banda[-1]] - 15:.1f} s"
        else:
            ts_approx = "< 1 s"
    print(f"  PM = {info['PM_deseado']}°: {'Estable' if estable else 'INESTABLE'}, "
          f"ts (post-falla) ≈ {ts_approx}")

### Observaciones B.c)

- A medida que se **incrementa el margen de fase**, el sistema se recupera mejor de la falla.
- Con PM bajo (35°), el sistema puede ser apenas estable o mostrar oscilaciones prolongadas.
- Con PM alto (65°), la respuesta es más amortiguada y robusta.
- El sistema **se parece cada vez más a un sistema de primer orden** (sobreamortiguado) a medida que el PM aumenta. Esto es coherente: mayor margen de fase → mayor amortiguamiento equivalente.

---

## B.d) Análisis de Nyquist compensado y desventajas

In [ ]:
# B.d.1) Diagrama de Nyquist comparativo

labels_nyq = [f'PM = {info["PM_deseado"]}°' for info in infos]

fig = graficar_nyquist_comparativo(G_nom, compensadores, labels_nyq,
    colores=['b', 'r', 'g'],
    titulo="B.d) Nyquist del sistema nominal compensado")
plt.show()

print("Observar cómo el compensador 'aleja' la curva del punto (-1, 0):")
print("  → Mayor PM = mayor distancia al punto crítico = mayor robustez.")

In [ ]:
# B.d.2) Análisis cuantitativo de las desventajas

print("=" * 65)
print("  TRADE-OFFS AL INCREMENTAR EL MARGEN DE FASE")
print("=" * 65)

for C, info in zip(compensadores, infos):
    CG = C * G_nom
    gm, pm, wcg, wcp = ctrl.margin(CG)
    # Ancho de banda (frecuencia de cruce)
    T = ctrl.feedback(CG, 1)
    t_step = np.linspace(0, 30, 3000)
    _, y = ctrl.step_response(T, t_step)
    y = np.array(y).flatten()
    
    # Sobrepico
    y_final = y[-1]
    Mp = (np.max(y) - y_final) / y_final * 100 if y_final > 0 else 0
    
    # Tiempo de establecimiento (2%)
    fuera = np.where(np.abs(y - y_final) > 0.02 * y_final)[0]
    ts = t_step[fuera[-1]] if len(fuera) > 0 else 0
    
    print(f"  PM = {info['PM_deseado']}°:  ωcp = {wcp:.2f} rad/s,  "
          f"Mp = {Mp:.1f}%,  ts = {ts:.1f} s,  "
          f"polo lead = {info['p']:.2f}")

print(f"{'='*65}")
print()
print("Desventajas de incrementar el MF:")
print("  1. MENOR ANCHO DE BANDA (ωcp decrece) → respuesta más lenta")
print("  2. MAYOR TIEMPO DE ESTABLECIMIENTO (ts)")
print("  3. POLO DEL COMPENSADOR más alejado → mayor esfuerzo de control")
print("  4. MENOR RECHAZO DE PERTURBACIONES en frecuencias medias")
print()
print("Conclusión: existe un trade-off entre ROBUSTEZ y DESEMPEÑO.")
print("El diseñador debe elegir el PM según el nivel de incertidumbre")
print("esperado en los parámetros de la planta.")

### Observaciones B.d)

**¿Cómo el compensador "aleja" la curva del punto crítico?**

El compensador lead aporta fase positiva en la frecuencia de cruce. Esto **rota** la curva de Nyquist en sentido antihorario cerca de $(-1, 0)$, aumentando la distancia mínima al punto crítico. A mayor margen de fase, mayor es esta rotación y más robusta es la operación frente a degradación de $B$.

**Desventajas de incrementar el MF indefinidamente:**

1. **Respuesta más lenta**: el ancho de banda disminuye.
2. **Mayor tiempo de establecimiento**: el sistema se vuelve sobreamortiguado.
3. **Compensador más "agresivo"**: polo del lead más alejado → más sensible a ruido en alta frecuencia.
4. **Peor rechazo de perturbaciones**: menor ganancia en la banda media.

> **En resumen**: no se puede incrementar el MF sin límite. Hay un compromiso óptimo entre robustez (MF alto) y desempeño dinámico (MF moderado). El valor adecuado depende del rango de variación esperado de los parámetros.